In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
folder_path = r"/storage/alplakes_test/neuchatel_100m_2025"
input_folder = os.path.join(folder_path, "outputs_swirl", "eddy_catalogues_final")

output_folder = os.path.join(folder_path, "outputs_swirl", "eddy_statistics")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
lvl0_csv_path = os.path.join(input_folder, "lvl0.csv")

In [ ]:
df_lvl0 = pd.read_csv(lvl0_csv_path)
df_lvl0 = df_lvl0.set_index('id', drop=False)
df_lvl0['date'] = pd.to_datetime(df_lvl0['date'])

In [ ]:
lake_mask = np.load(os.path.join(folder_path, "grid", "mask_lake.npy"))

In [ ]:
depths = df_lvl0[['depth_index', 'depth_[m]']].drop_duplicates().sort_values('depth_index').reset_index(drop=True)

In [ ]:
Nx = 390 # lucerne : 288, geneva: 660
Ny = 96 # lucerne : 180, geneva: 245

# Volume in upper layer

In [ ]:
df_lvl0_upper_layer = df_lvl0[df_lvl0['depth_index'] < 16]

In [ ]:
volume_timeserie_upper_layer = df_lvl0_upper_layer.groupby(['date'])['volume_slice_[m3]'].sum()

In [ ]:
(volume_timeserie_upper_layer/1e6).to_csv(os.path.join(output_folder, 'volume_timeserie_upper_layer.csv'))

# Snapshot eddies

In [ ]:
def get_eddy_cells_snapshot(df, i_time, i_depth_min, i_depth_max, Nx, Ny):
    depth_filter = (
        (df['depth_index'] <= i_depth_max) &
        (df['depth_index'] >= i_depth_min)
    )
    time_filter = df['time_index'] == i_time

    df_filtered = df.loc[depth_filter & time_filter]

    if len(df_filtered) == 0:
        return np.zeros((Nx, Ny)).T

    i_eddies = []
    j_eddies = []
    for _, row in df_filtered.iterrows():
        i = np.fromstring(row['i_eddy_cells'].strip("[]"), sep=',')
        j = np.fromstring(row['j_eddy_cells'].strip("[]"), sep=',')

        i_eddies.append(i)
        j_eddies.append(j)

    i_eddies = np.concat(i_eddies)
    j_eddies = np.concat(j_eddies)

    df_eddy_cells_snapshot = pd.DataFrame({
                                    'i_eddy_cells': i_eddies.astype(int),
                                    'j_eddy_cells': j_eddies.astype(int)
                                }).drop_duplicates()

    img = np.zeros((Nx, Ny))
    img[df_eddy_cells_snapshot["i_eddy_cells"], df_eddy_cells_snapshot["j_eddy_cells"]] = 1

    return img.T

In [ ]:
i_time = 2329
i_depth_min = 21
i_depth_max = 24

In [ ]:
eddy_cells_snapshot = get_eddy_cells_snapshot(df_lvl0, i_time, i_depth_min, i_depth_max, Nx, Ny)

In [ ]:
plt.imshow(eddy_cells_snapshot, origin="lower")

# Statistics per location

In [ ]:
depths

In [ ]:
i_depth_min = 0
i_depth_max = 8
Nt = int(df_lvl0['time_index'].max())
start_idx = 30*24*4
end_idx = 24*(30*10+4)
figsize=(15,5)

In [ ]:
np.unique(df_lvl0['date'])[24*(30*10+4)]

In [ ]:
str_depth_max = str(round(depths.iloc[i_depth_max]['depth_[m]'], 2))
str_depth_min = str(round(depths.iloc[i_depth_min]['depth_[m]'], 2))

In [ ]:
eddy_nb = np.zeros((Nx, Ny)).T
for i_idx in range(start_idx,end_idx):
    eddy_cells_snapshot = get_eddy_cells_snapshot(df_lvl0, i_idx, i_depth_min, i_depth_max, Nx, Ny)
    eddy_nb += eddy_cells_snapshot

eddy_perc = 100 * eddy_nb / (Nt+1)
eddy_perc[~lake_mask[i_depth_min]] = np.nan

In [ ]:
plt.figure(figsize=figsize)
plt.imshow(eddy_perc, cmap='jet', origin="lower", vmin=0, vmax=20)
plt.title("Probability of eddy occurrence (mai-october)")
cbar = plt.colorbar()
cbar.set_label("Eddy probability [%]")
plt.text(0.98, 0.98, f'{str_depth_min}m to {str_depth_max}m', transform=plt.gca().transAxes, ha='right', va='top')
plt.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_folder, f"eddy_probability_map_{str_depth_min}m_to_{str_depth_max}m_stratified.png"))

## Clockwise

In [ ]:
df_clockwise = df_lvl0[df_lvl0['rotation_direction']=='clockwise']
eddy_cw_nb = np.zeros((Nx, Ny)).T
for i_idx in range(start_idx,end_idx):
    eddy_cells_snapshot = get_eddy_cells_snapshot(df_clockwise, i_idx, i_depth_min, i_depth_max, Nx, Ny)
    eddy_cw_nb += eddy_cells_snapshot

eddy_cw_perc = 100 * eddy_cw_nb / (Nt+1)
eddy_cw_perc[~lake_mask[i_depth_min]] = np.nan

In [ ]:
plt.figure(figsize=figsize)
plt.imshow(eddy_cw_perc, cmap='jet', origin="lower", vmin=0, vmax=20)
plt.title("Probability of clockwise eddy occurrence (mai-october)")
cbar = plt.colorbar()
cbar.set_label("Eddy probability [%]")
plt.text(0.98, 0.98, f'{str_depth_min}m to {str_depth_max}m', transform=plt.gca().transAxes, ha='right', va='top')
plt.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_folder, f"cw_eddy_probability_map_{str_depth_min}m_to_{str_depth_max}m_stratified.png"))

## Anti-clockwise

In [ ]:
df_acw = df_lvl0[df_lvl0['rotation_direction']=='anticlockwise']
eddy_acw_nb = np.zeros((Nx, Ny)).T
for i_idx in range(start_idx,end_idx):
    eddy_cells_snapshot = get_eddy_cells_snapshot(df_acw, i_idx, i_depth_min, i_depth_max, Nx, Ny)
    eddy_acw_nb += eddy_cells_snapshot

eddy_acw_perc = 100 * eddy_acw_nb / (Nt+1)
eddy_acw_perc[~lake_mask[i_depth_min]] = np.nan

In [ ]:
plt.figure(figsize=figsize)
plt.imshow(eddy_acw_perc, cmap='jet', origin="lower", vmin=0, vmax=20)
plt.title("Probability of anti-clockwise eddy occurrence (mai-october)")
cbar = plt.colorbar()
cbar.set_label("Eddy probability [%]")
plt.text(0.98, 0.98, f'{str_depth_min}m to {str_depth_max}m', transform=plt.gca().transAxes, ha='right', va='top')
plt.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_folder, f"acw_eddy_probability_map_{str_depth_min}m_to_{str_depth_max}m_stratified.png"))